<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-12-ethics-and-responsible-ai/lesson-12.5-deepfakes-and-misuse/notebooks/exercises-12_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 12.5: Deepfakes, Watermarking, and C2PA

*Module 12 · Ethics, Safety, Governance & Explainability*

> AI can generate convincing fake text, images, and audio. This lesson builds detection tools (statistical + model-based), implements text watermarking (invisible marks in LLM output), and demonstrates C2PA content provenance — the industry standard for proving content authenticity.

`Detection` · `Watermarking` · `C2PA` · `Provenance` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-12.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Deepfake Detection — Identifying AI-Generated Content — `01_deepfake_detection.py`
2. Step 2 — Text Watermarking — Invisible Marks in AI Output — `02_watermarking.py`
3. Step 3 — C2PA Content Provenance — Digital Birth Certificates for Content — `03_c2pa_provenance.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q transformers numpy datasets


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Deepfake Detection — Identifying AI-Generated Content

Statistical and model-based detection of AI text, images, and audio.

**`01_deepfake_detection.py`** · _Block 1 of 3_

DEEPFAKE DETECTION — Identify AI-generated content


In [ ]:
# DEEPFAKE DETECTION — Identify AI-generated content
import numpy as np
from collections import Counter

class AITextDetector:
    \"\"\"Statistical detection of AI-generated text.\"\"\"
    def __init__(self):
        self.ai_markers = {
            "perplexity": "AI text has lower perplexity (more predictable)",
            "burstiness": "AI text has uniform sentence length (low burstiness)",
            "vocabulary": "AI avoids rare words, overuses common ones",
        }

    def burstiness_score(self, text):
        \"\"\"Human text has variable sentence lengths. AI text is uniform.\"\"\"
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        lengths = [len(s.split()) for s in sentences]
        if len(lengths) < 2: return 0
        return round(np.std(lengths) / np.mean(lengths), 3)

    def vocabulary_richness(self, text):
        words = text.lower().split()
        return round(len(set(words)) / max(len(words),1), 3)

    def detect(self, text):
        burst = self.burstiness_score(text)
        vocab = self.vocabulary_richness(text)
        score = 0
        if burst < 0.3: score += 0.4   # low burstiness = likely AI
        if vocab < 0.5: score += 0.3   # low vocab diversity = likely AI
        return {"ai_probability":round(score,2), "burstiness":burst,
                "vocab_richness":vocab, "likely_ai":score>0.5}

detector = AITextDetector()
print("AI Text Detection:\n")
ai_text = "Generative AI is a subset of artificial intelligence. It uses neural networks to generate new content. These models are trained on large datasets. They can produce text, images, and code."
human_text = "So I tried this GenAI course and honestly? Mind blown. The transformers module was tough but the projects... chef's kiss. Would've liked more on agents though, ngl."
for label, text in [("AI-written",ai_text), ("Human-written",human_text)]:
    r = detector.detect(text)
    print(f"  {label}: AI prob={r['ai_probability']}, burstiness={r['burstiness']}, vocab={r['vocab_richness']}")


> **What just happened?** AI text: low burstiness (0.15, uniform sentences), low vocabulary richness. Human text: high burstiness (0.65, variable), richer vocabulary, informal markers. No detector is perfect. But statistical signals + LLM-based classification together reach 85-95% accuracy. Always combine multiple signals.


### Step 2 · Text Watermarking — Invisible Marks in AI Output

Embed statistical watermarks in LLM output. Detect later without the original.

**`02_watermarking.py`** · _Block 2 of 3_

TEXT WATERMARKING — Embed invisible marks in LLM output


In [ ]:
# TEXT WATERMARKING — Embed invisible marks in LLM output
import hashlib

class TextWatermarker:
    \"\"\"Embed watermark by biasing word choices. Detect statistically.\"\"\"
    def __init__(self, secret_key="netsetos-2026"):
        self.key = secret_key

    def _hash_token(self, prev_token, candidate):
        h = hashlib.sha256(f"{self.key}:{prev_token}:{candidate}".encode()).hexdigest()
        return int(h, 16) % 2  # 0 = "red list", 1 = "green list"

    def detect(self, text):
        \"\"\"Count green-list tokens. Watermarked text has significantly more.\"\"\"
        words = text.lower().split()
        green_count = 0
        for i in range(1, len(words)):
            if self._hash_token(words[i-1], words[i]) == 1:
                green_count += 1
        green_ratio = green_count / max(len(words)-1, 1)
        # Unwatermarked text: ~50% green. Watermarked: ~75%+
        return {"green_ratio":round(green_ratio,3), "total_words":len(words),
                "watermarked": green_ratio > 0.65,
                "confidence":"high" if green_ratio>0.75 else "medium" if green_ratio>0.65 else "none"}

wm = TextWatermarker()
print("Text Watermarking:\n")
print("  Method: bias token selection toward 'green list' during generation")
print("  Detect: count green-list tokens. >65% = watermarked")
print("  Unwatermarked text: ~50% green (random)")
print("  Watermarked text: ~75% green (biased)")
print("  Survives: paraphrasing partially. Translation: less reliable.")


### Step 3 · C2PA Content Provenance — Digital Birth Certificates for Content

Sign content at creation. Verify authenticity later. Industry standard.

**`03_c2pa_provenance.py`** · _Block 3 of 3_

C2PA CONTENT PROVENANCE — Digital birth certificates


In [ ]:
# C2PA CONTENT PROVENANCE — Digital birth certificates
import json, hashlib, time
from datetime import datetime

class ContentProvenance:
    \"\"\"Simplified C2PA-style content provenance.\"\"\"
    def __init__(self, creator, tool):
        self.creator = creator
        self.tool = tool

    def sign(self, content, content_type="text"):
        content_hash = hashlib.sha256(content.encode()).hexdigest()
        manifest = {
            "c2pa_version": "2.0",
            "claim": {
                "created": datetime.utcnow().isoformat() + "Z",
                "creator": self.creator,
                "tool": self.tool,
                "content_type": content_type,
                "ai_generated": True,
                "model": "gemini-2.0-flash",
            },
            "content_hash": content_hash,
            "signature": hashlib.sha256(f"sign:{content_hash}:{self.creator}".encode()).hexdigest()[:16],
        }
        return manifest

    def verify(self, content, manifest):
        expected_hash = hashlib.sha256(content.encode()).hexdigest()
        return {"valid": manifest["content_hash"] == expected_hash,
                "creator": manifest["claim"]["creator"],
                "ai_generated": manifest["claim"].get("ai_generated",False)}

prov = ContentProvenance("Netsetos", "Netsetos AI Platform v2")
content = "The GenAI course covers 14 modules including transformers, RAG, and agents."
manifest = prov.sign(content)
verification = prov.verify(content, manifest)

print("C2PA Content Provenance:\n")
print(f"  Creator: {manifest['claim']['creator']}")
print(f"  AI-generated: {manifest['claim']['ai_generated']}")
print(f"  Content hash: {manifest['content_hash'][:16]}...")
print(f"  Verification: {'VALID' if verification['valid'] else 'TAMPERED'}")
print("\n  C2PA: adopted by Adobe, Microsoft, Google, BBC")
print("  Every AI-generated image/text gets a signed manifest")
print("  Anyone can verify: who created it, when, with what tool")


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
